# Real Estate House Price Prediction — Gradient Boosting Regression

**Business Objective:** Deliver automated, accurate property market valuations using physical architectural dimensions, structural age, location tier, and neighborhood amenities.

### Valuation Modeling Workflow
```text
Residential Property Data (house_price_data.csv)
                    ↓
Exploratory Data Analysis (EDA) & Feature Correlation
                    ↓
Feature Pipeline (One-Hot Encoding & StandardScaler)
                    ↓
Gradient Boosting Regressor Training
                    ↓
Performance Evaluation (R², RMSE, MAE, Residuals)
                    ↓
Feature Importance Analysis
                    ↓
Interactive Property Price Calculator
```

This notebook is fully self-contained and ready to execute in Google Colab or local Jupyter.


## 1. Install & import libraries
Run this cell first. All required packages are imported for statistical modeling, visualization, and interactive widgets.


In [ ]:
!pip install -q numpy pandas matplotlib seaborn scikit-learn ipywidgets

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

# Styling
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["font.size"] = 10
print("Libraries imported successfully!")


## 2. Load and inspect the dataset
We load `house_price_data.csv` containing 1,500 residential properties and verify structure, summary statistics, and column types.


In [ ]:
# Locate dataset file
csv_candidates = [
    "house_price_data.csv",
    "house-price-prediction-model--main/house_price_data.csv",
    "../house_price_data.csv"
]
csv_path = next((p for p in csv_candidates if os.path.exists(p)), "house_price_data.csv")

df = pd.read_csv(csv_path)
print(f"Dataset shape: {df.shape[0]} properties, {df.shape[1]} features\n")
print("First 5 records:")
display(df.head())

print("\nMissing values check:")
print(df.isnull().sum())

print("\nDescriptive statistics:")
display(df.describe().round(2))


## 3. Exploratory Data Analysis (EDA)
Examining the distribution of property prices, spatial location trends, and correlations across property attributes.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Price Distribution
sns.histplot(df['price_in_lakhs'], kde=True, color='#1E88E5', bins=30, ax=axes[0, 0])
axes[0, 0].set_title('Property Price Distribution (in ₹ Lakhs)', fontsize=12, fontweight='bold')
axes[0, 0].set_xlabel('Price (₹ Lakhs)')
axes[0, 0].set_ylabel('Property Count')

# 2. Square Feet vs Price by Location
palette = {'Urban': '#D81B60', 'Suburban': '#1E88E5', 'Rural': '#004D40'}
sns.scatterplot(
    data=df,
    x='square_feet',
    y='price_in_lakhs',
    hue='location_type',
    palette=palette,
    alpha=0.6,
    ax=axes[0, 1]
)
axes[0, 1].set_title('Price vs Built-up Area (Sq Ft) by Location', fontsize=12, fontweight='bold')
axes[0, 1].set_xlabel('Square Feet')
axes[0, 1].set_ylabel('Price (₹ Lakhs)')

# 3. Correlation Heatmap
numeric_cols = df.select_dtypes(include=[np.number]).columns
corr = df[numeric_cols].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', cbar=True, ax=axes[1, 0])
axes[1, 0].set_title('Feature Correlation Matrix', fontsize=12, fontweight='bold')

# 4. Bedroom Count vs Price by Garden Availability
sns.boxplot(
    data=df,
    x='bedrooms',
    y='price_in_lakhs',
    hue='has_garden',
    palette={'Yes': '#43A047', 'No': '#E53935'},
    ax=axes[1, 1]
)
axes[1, 1].set_title('Price by Bedrooms & Private Garden', fontsize=12, fontweight='bold')
axes[1, 1].set_xlabel('Bedrooms')
axes[1, 1].set_ylabel('Price (₹ Lakhs)')

plt.tight_layout()
plt.show()


## 4. Preprocessing & Train/Test Split
We separate target variable (`price_in_lakhs`), drop unique ID, and split the data into 80% training and 20% testing sets.


In [ ]:
X = df.drop(columns=['property_id', 'price_in_lakhs'])
y = df['price_in_lakhs']

categorical_cols = ['location_type', 'has_garden']
numerical_cols = [
    'square_feet', 'bedrooms', 'bathrooms', 'property_age_years',
    'garage_spaces', 'distance_to_city_center_km', 'school_rating'
]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

print(f"Training samples: {X_train.shape[0]}")
print(f"Testing samples:  {X_test.shape[0]}")


## 5. Build Regression Pipeline & Model Training
We build a modular `ColumnTransformer` with `StandardScaler` and `OneHotEncoder`. We then train a **Gradient Boosting Regressor** and benchmark it against **Random Forest** and **Ridge Regression**.


In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_cols),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_cols)
    ]
)

# Define models
models = {
    "Gradient Boosting": GradientBoostingRegressor(n_estimators=200, learning_rate=0.08, max_depth=4, random_state=42),
    "Random Forest": RandomForestRegressor(n_estimators=150, max_depth=10, random_state=42, n_jobs=-1),
    "Ridge Regression": Ridge(alpha=1.0)
}

trained_models = {}
for name, model in models.items():
    pipe = Pipeline(steps=[('preprocessor', preprocessor), ('regressor', model)])
    pipe.fit(X_train, y_train)
    trained_models[name] = pipe

print("All regression models trained successfully!")


## 6. Model Evaluation & Benchmark
We evaluate all three models using $R^2$ (variance explained), RMSE (Root Mean Squared Error in Lakhs), and MAE (Mean Absolute Error).


In [ ]:
results = []
for name, pipe in trained_models.items():
    y_pred = pipe.predict(X_test)
    r2 = r2_score(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)
    results.append({"Model": name, "R² Score": r2, "RMSE (₹ Lakhs)": rmse, "MAE (₹ Lakhs)": mae})

eval_df = pd.DataFrame(results).sort_values("R² Score", ascending=False)
display(eval_df.style.format({"R² Score": "{:.4f}", "RMSE (₹ Lakhs)": "{:.2f}", "MAE (₹ Lakhs)": "{:.2f}"}))

best_model = trained_models["Gradient Boosting"]
best_pred = best_model.predict(X_test)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# 1. Actual vs Predicted Prices
axes[0].scatter(y_test, best_pred, alpha=0.5, color='#1976D2', edgecolors='k', s=40)
min_val = min(y_test.min(), best_pred.min())
max_val = max(y_test.max(), best_pred.max())
axes[0].plot([min_val, max_val], [min_val, max_val], color='#D32F2F', lw=2.5, linestyle='--', label='Ideal 45° Fit')
axes[0].set_title('Actual vs Predicted Valuation', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Actual Price (₹ Lakhs)')
axes[0].set_ylabel('Predicted Price (₹ Lakhs)')
axes[0].legend()

# 2. Residual Distribution
residuals = y_test - best_pred
sns.histplot(residuals, kde=True, color='#388E3C', bins=25, ax=axes[1])
axes[1].axvline(0, color='red', linestyle='--', lw=2)
axes[1].set_title('Residual Error Distribution (Actual - Predicted)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Valuation Error (₹ Lakhs)')
axes[1].set_ylabel('Frequency')

plt.tight_layout()
plt.show()


## 7. Which features matter most? (Feature Importance)
We extract feature importances from the Gradient Boosting Regressor to isolate key drivers of residential valuation.


In [ ]:
cat_encoder = best_model.named_steps['preprocessor'].named_transformers_['cat']
cat_features = list(cat_encoder.get_feature_names_out(categorical_cols))
all_features = numerical_cols + cat_features

importances = best_model.named_steps['regressor'].feature_importances_
feat_df = pd.DataFrame({
    'Feature': all_features,
    'Importance': importances
}).sort_values('Importance', ascending=True)

plt.figure(figsize=(10, 6))
plt.barh(feat_df['Feature'], feat_df['Importance'], color='#0288D1', edgecolor='black')
plt.title('Gradient Boosting Feature Importance in Property Pricing', fontsize=12, fontweight='bold')
plt.xlabel('Relative Importance')
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

print("Top 5 Valuation Determinants:")
print(feat_df.tail(5).iloc[::-1].to_string(index=False))


## 8. Predict Valuation for a Specific Property
We pass a hypothetical residential listing to generate an estimated valuation along with a 90% confidence range.


In [ ]:
sample_property = pd.DataFrame([{
    'square_feet': 2100,
    'bedrooms': 3,
    'bathrooms': 2,
    'location_type': 'Urban',
    'property_age_years': 5,
    'garage_spaces': 2,
    'has_garden': 'Yes',
    'distance_to_city_center_km': 4.5,
    'school_rating': 8.5
}])

pred_price = best_model.predict(sample_property)[0]
lower_bound = pred_price * 0.95
upper_bound = pred_price * 1.05

print("=== Property Valuation Assessment ===")
display(sample_property)
print(f"\nEstimated Fair Market Price: ₹{pred_price:.2f} Lakhs")
print(f"Estimated Valuation Range:   ₹{lower_bound:.2f} Lakhs to ₹{upper_bound:.2f} Lakhs")


## 9. Interactive Property Price Estimator (Live UI Sliders)
Adjust property dimensions, room counts, and neighborhood attributes below to calculate instant real estate market values.


In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

sqft_w = widgets.IntSlider(value=1800, min=600, max=4500, step=50, description='Sq Feet:')
beds_w = widgets.IntSlider(value=3, min=1, max=5, step=1, description='Bedrooms:')
baths_w = widgets.IntSlider(value=2, min=1, max=4, step=1, description='Bathrooms:')
loc_w = widgets.Dropdown(options=['Urban', 'Suburban', 'Rural'], value='Suburban', description='Location:')
age_w = widgets.IntSlider(value=8, min=0, max=35, step=1, description='Age (Years):')
garage_w = widgets.IntSlider(value=1, min=0, max=3, step=1, description='Garage:')
garden_w = widgets.Dropdown(options=['Yes', 'No'], value='Yes', description='Garden:')
dist_w = widgets.FloatSlider(value=10.0, min=1.0, max=30.0, step=0.5, description='CBD Dist (km):')
school_w = widgets.FloatSlider(value=7.5, min=2.0, max=10.0, step=0.5, description='School:')

out = widgets.Output()

def estimate_valuation(b=None):
    with out:
        clear_output()
        prop = pd.DataFrame([{
            'square_feet': sqft_w.value,
            'bedrooms': beds_w.value,
            'bathrooms': baths_w.value,
            'location_type': loc_w.value,
            'property_age_years': age_w.value,
            'garage_spaces': garage_w.value,
            'has_garden': garden_w.value,
            'distance_to_city_center_km': dist_w.value,
            'school_rating': school_w.value
        }])
        
        val = best_model.predict(prop)[0]
        
        print("=" * 55)
        print(f"ESTIMATED PROPERTY VALUATION: ₹{val:.2f} Lakhs")
        print(f"Valuation Range (±5%):        ₹{val*0.95:.2f} - ₹{val*1.05:.2f} Lakhs")
        print(f"Implied Price per Sq Ft:      ₹{int((val * 100000) / sqft_w.value):,}/sqft")
        print("=" * 55)

btn = widgets.Button(description="Calculate Valuation", button_style="success")
btn.on_click(estimate_valuation)

ui = widgets.VBox([
    widgets.HBox([sqft_w, loc_w]),
    widgets.HBox([beds_w, baths_w]),
    widgets.HBox([age_w, garage_w]),
    widgets.HBox([garden_w, dist_w]),
    school_w,
    btn,
    out
])

display(ui)
estimate_valuation()
